# 03 - ViBERT (FPTAI) base model: 5-fold OOF + full-train refit (seed=42)

Outputs `artifacts/probs/vibert_{oof,val,test}.npy` and `artifacts/metrics/vibert.json`.

Hyperparameters follow the CafeBERT fine-tuning setup from the thesis while keeping 5-fold OOF.

In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy

In [2]:
import sys, os

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

Mounted at /content/drive
[utils_io] pulled 128 files: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts -> /content/ensemble_tmp/artifacts
[colab_setup] is_colab=True
[colab_setup] repo_root=/content/drive/MyDrive/thesis/topicmodeling
[colab_setup] tmp_root=/content/ensemble_tmp (hf_cache/bert_work/lora_work)
[colab_setup] artifacts=/content/ensemble_tmp/artifacts
[colab_setup] persistent_artifacts=/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts
[colab_setup] HF_TOKEN: set


In [3]:
from tm_research.ensemble.utils_io import load_splits
from tm_research.ensemble.bert_oof import BertOOFConfig, run_bert_oof

train_df, val_df, test_df, label_map = load_splits()
print('train', train_df.shape, 'val', val_df.shape, 'test', test_df.shape)
print('classes', label_map.class_names)

train (10549, 2) val (686, 2) test (692, 2)
classes ['Anger', 'Disgust', 'Enjoyment', 'Fear', 'Other', 'Sadness', 'Surprise']


In [4]:
RUN_SEED = 42
print(f'Using fixed seed: {RUN_SEED}')

cfg = BertOOFConfig(
    model_name='FPTAI/vibert-base-cased',
    output_name='vibert',
    seed=RUN_SEED,
    n_folds=3,
    num_epochs=20,
    learning_rate=2e-5,
    max_length=256,
    train_batch_size=128,
    eval_batch_size=256,
    grad_accum_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    early_stopping_patience=3,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    work_dir=str(paths.bert_work / 'vibert'),
)
metrics = run_bert_oof(cfg, train_df, val_df, test_df, label_map)
metrics

Using fixed seed: 42
[vibert] running 3-fold OOF on N=10549


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  581MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/692 [00:00<?, ? examples/s]

[vibert] done in 7.2 min — oof=0.5823 val=0.5466 test=0.5303


{'model_name': 'FPTAI/vibert-base-cased',
 'output_name': 'vibert',
 'seed': 42,
 'n_folds': 3,
 'fold_val_accuracies': [0.5928348023883993,
  0.593287827076223,
  0.5608646188850968],
 'oof_accuracy': 0.5823300786804436,
 'oof_recall': 0.5815018295697287,
 'oof_f1': 0.5481945389338538,
 'oof_f1_weighted': 0.5486767588457943,
 'val_accuracy': 0.5466472303206997,
 'val_recall': 0.5133599934266134,
 'val_f1': 0.5051386597802505,
 'val_f1_weighted': 0.5526941601354423,
 'test_accuracy': 0.5303468208092486,
 'test_recall': 0.484857847790337,
 'test_f1': 0.4912260254120251,
 'test_f1_weighted': 0.5367668584567552,
 'num_train': 10549,
 'num_val': 686,
 'num_test': 692,
 'elapsed_seconds': 429.8706545829773,
 'tuning': {'learning_rate': 2e-05,
  'lr_scheduler_type': 'cosine',
  'warmup_ratio': 0.1,
  'weight_decay': 0.01,
  'num_epochs': 20,
  'train_batch_size': 128,
  'grad_accum_steps': 2,
  'max_length': 256,
  'early_stopping_patience': 3,
  'max_grad_norm': 1.0}}

In [5]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()

[utils_io] pushed 128 files: /content/ensemble_tmp/artifacts -> /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts


PosixPath('/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts')

In [6]:
from google.colab import runtime
runtime.unassign()